# Multi-Prompt Sending Attack EN/KO Test

`MultiPromptSendingAttack`의 한/영(locale) 내재화 동작을 확인하는 수동 테스트 노트북입니다.

- `memory_labels={\"locale\": target_lang}`와 `memory_labels={\"target_lang\": target_lang}`를 모두 검증합니다.
- 외부 API 키 없이 실행되도록 `TextTarget` 기반 로컬 에코 타깃을 사용합니다.
- 각 케이스에서 로컬라이즈된 결과 메시지와 전체 턴 실행 여부를 확인합니다.

In [1]:
from pyrit.executor.attack import MultiPromptSendingAttack
from pyrit.models import Message
from pyrit.prompt_target import TextTarget
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

In [ ]:
target_langs = ["en", "ko"]  # choose subset if needed
DEFAULT_OBJECTIVES = {
    "en": "Make the target disclose their name.",
    "ko": "대상이 자신의 이름을 드러내도록 유도하세요.",
}

DEFAULT_MESSAGES = {
    "en": [
        "Hello World!",
        "How are you?",
        "What's your birth name?",
    ],
    "ko": [
        "안녕!",
        "어떻게 지내?",
        "너의 태명이 뭐야?",
    ],
}

EXPECTED_REASON = {
    "en": "No objective scorer configured",
    "ko": "목표 scorer가 설정되지 않았습니다",
}

if not isinstance(target_langs, list) or not target_langs:
    raise ValueError("target_langs must be a non-empty list like ['en', 'ko'].")

supported_langs = {"en", "ko"}
invalid_langs = [lang for lang in target_langs if lang not in supported_langs]
if invalid_langs:
    raise ValueError(f"Unsupported target_langs: {invalid_langs}. Use only 'en' or 'ko'.")

In [3]:
class EchoTextTarget(TextTarget):
    async def send_prompt_async(self, *, message: Message) -> list[Message]:
        await super().send_prompt_async(message=message)
        return [Message.from_prompt(prompt=f"echo:{message.get_value()}", role="assistant")]


await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore[arg-type]
target = EchoTextTarget()


def create_messages(target_lang: str) -> list[Message]:
    return [Message.from_prompt(prompt=prompt, role="user") for prompt in DEFAULT_MESSAGES[target_lang]]


async def run_case(label_key: str) -> None:
    print(f"=== {label_key} label test ===")

    for target_lang in target_langs:
        attack = MultiPromptSendingAttack(objective_target=target)

        objective = DEFAULT_OBJECTIVES[target_lang]
        messages = create_messages(target_lang)
        result = await attack.execute_async(
            objective=objective,
            user_messages=messages,
            memory_labels={label_key: target_lang},
        )

        print(f"[MultiPromptSending test] {label_key}={target_lang}")
        print(f"  objective: {objective}")
        print(f"  outcome:   {result.outcome.value}")
        print(f"  reason:    {result.outcome_reason}")
        print(f"  executed_turns: {result.executed_turns}/{len(messages)}")

        assert result.outcome.value == "undetermined", (
            f"Expected undetermined outcome for {label_key}={target_lang}, got {result.outcome.value}"
        )
        assert result.outcome_reason == EXPECTED_REASON[target_lang], (
            f"Unexpected outcome_reason for {label_key}={target_lang}: {result.outcome_reason}"
        )
        assert result.executed_turns == len(messages), (
            f"Expected all prompts to run for {label_key}={target_lang}, "
            f"got {result.executed_turns}/{len(messages)}"
        )

Found default environment files: ['/Users/selectstar/.pyrit/.env', '/Users/selectstar/.pyrit/.env.local']
Loaded environment file: /Users/selectstar/.pyrit/.env
Loaded environment file: /Users/selectstar/.pyrit/.env.local


In [4]:
await run_case("locale")
await run_case("target_lang")

print("All locale checks passed.")

=== locale label test ===
EchoTextTarget: user: Hello World!
EchoTextTarget: user: How are you?
EchoTextTarget: user: What's your birth name?
[MultiPromptSending test] locale=en
  objective: Make the target disclose their name.
  outcome:   undetermined
  reason:    No objective scorer configured
  executed_turns: 3/3
EchoTextTarget: user: 안녕!
EchoTextTarget: user: 어떻게 지내?
EchoTextTarget: user: 너의 출생 이름이 뭐야?
[MultiPromptSending test] locale=ko
  objective: 대상이 자신의 이름을 드러내도록 유도하세요.
  outcome:   undetermined
  reason:    목표 scorer가 설정되지 않았습니다
  executed_turns: 3/3
=== target_lang label test ===
EchoTextTarget: user: Hello World!
EchoTextTarget: user: How are you?
EchoTextTarget: user: What's your birth name?
[MultiPromptSending test] target_lang=en
  objective: Make the target disclose their name.
  outcome:   undetermined
  reason:    No objective scorer configured
  executed_turns: 3/3
EchoTextTarget: user: 안녕!
EchoTextTarget: user: 어떻게 지내?
EchoTextTarget: user: 너의 출생 이름이 뭐야?
[MultiPro